## Engenharia de Variaveis

Constroi o dataset final para os modelos de deteccao de anomalias (Isolation Forest, LOF, One-Class SVM).

Entradas:
- `data/raw/propostas_pe.csv` — propostas por item de licitacao
- `data/raw/fornecedores_cnpj.csv` — dados cadastrais dos fornecedores
- `data/raw/municipios_pe.csv` — coordenadas dos municipios
- `data/raw/licitacoes_pe.csv` — metadados das contratacoes

Saida: `data/processed/dataset_ml.csv`

In [1]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime, date

os.makedirs("data/processed", exist_ok=True)
print("Setup OK")

Setup OK


### 1. Carrega e Inspeciona os Dados Brutos

In [2]:
propostas  = pd.read_csv("data/raw/propostas_pe.csv",          dtype=str)
licitacoes = pd.read_csv("data/raw/contratacoes14133_pe.csv",  dtype=str)
cnpj_df    = pd.read_csv("data/raw/fornecedores_cnpj.csv",     dtype=str)
municipios = pd.read_csv("data/raw/municipios_pe.csv",         dtype=str)

print(f"Propostas:   {len(propostas)} linhas | colunas: {list(propostas.columns)}")
print(f"Licitações:  {len(licitacoes)} linhas | colunas: {list(licitacoes.columns[:8])}...")
print(f"CNPJ:        {len(cnpj_df)} linhas")
print(f"Municípios:  {len(municipios)} linhas")

Propostas:   75502 linhas | colunas: ['idCompra', 'orgaoEntidadeCnpj', 'anoCompraPncp', 'sequencialCompraPncp', 'numeroItem', 'descricaoItem', 'valor_unitario_estimado', 'valor_unitario_homologado', 'cnpj_fornecedor', 'nome_fornecedor', 'valor_total_homologado', 'situacao_resultado']
Licitações:  11353 linhas | colunas: ['idCompra', 'numeroControlePNCP', 'anoCompraPncp', 'sequencialCompraPncp', 'orgaoEntidadeCnpj', 'orgaoSubrogadoCnpj', 'codigoOrgao', 'orgaoEntidadeRazaoSocial']...
CNPJ:        9458 linhas
Municípios:  185 linhas


### 2. Join: Propostas + Metadados da Licitacao

In [3]:
JOIN_KEYS = ["orgaoEntidadeCnpj", "anoCompraPncp", "sequencialCompraPncp"]

for k in JOIN_KEYS:
    propostas[k]  = propostas[k].astype(str).str.strip()
    licitacoes[k] = licitacoes[k].astype(str).str.strip()

COLS_LIT = [
    *JOIN_KEYS,
    "unidadeOrgaoCodigoIbge",   # código IBGE do município do órgão
    "unidadeOrgaoMunicipioNome",# nome do município
    "modalidadeNome",
    "valorTotalEstimado",
    "dataPublicacaoPncp",
    "dataAberturaPropostaPncp",
]
cols_lit = [c for c in COLS_LIT if c in licitacoes.columns]

df = propostas.merge(
    licitacoes[cols_lit].drop_duplicates(subset=JOIN_KEYS),
    on=JOIN_KEYS,
    how="left"
)
print(f"Após join com licitações: {len(df)} linhas")
print(f"Colunas: {list(df.columns)}")

Após join com licitações: 75502 linhas
Colunas: ['idCompra', 'orgaoEntidadeCnpj', 'anoCompraPncp', 'sequencialCompraPncp', 'numeroItem', 'descricaoItem', 'valor_unitario_estimado', 'valor_unitario_homologado', 'cnpj_fornecedor', 'nome_fornecedor', 'valor_total_homologado', 'situacao_resultado', 'unidadeOrgaoCodigoIbge', 'unidadeOrgaoMunicipioNome', 'modalidadeNome', 'valorTotalEstimado', 'dataPublicacaoPncp', 'dataAberturaPropostaPncp']


### 3. Join: + Dados CNPJ do Fornecedor

O cadastro (`fornecedores_cnpj.csv`) é normalizado para 14 dígitos antes do *join*. Treze linhas dele são identificadores que não são CNPJ
(`000ESTRANGEIRO`, `0000000000MDPI`, `https:wwwiojtorg`…), todas com erro de consulta e sem dados cadastrais; ao tirar os caracteres não numéricos,
onze viram `00000000000000` e duas viram `00000000000001`. Com o cadastro repetido nessas chaves, o *join* anterior **repetia cada proposta desses fornecedores
(cerca de 11 vezes)** e a base crescia de 75.502 para 76.218 linhas (+716, todas contadas como pessoa física).
Agora o cadastro é deduplicado pela chave normalizada (mantendo, se houver, a linha sem erro) e o *join* é validado como muitos-para-um, sem alterar o número de linhas.

**Limitação mantida:** fornecedores estrangeiros ou sem CNPJ continuam agrupados em uma chave comum (`00000000000000`) e classificados como `pessoa_fisica` (o prefixo `000` vale para CPF e também para esses identificadores);
são poucas propostas (ver a saída da célula abaixo), sem dados cadastrais.

In [4]:
cnpj_forn_col = "cnpj_fornecedor"

cnpj_df["cnpj_norm"] = cnpj_df["cnpj"].astype(str).str.replace(r"\D", "", regex=True).str.zfill(14)
df["cnpj_forn_norm"] = df[cnpj_forn_col].astype(str).str.replace(r"\D", "", regex=True).str.zfill(14)

df["tipo_fornecedor"] = df["cnpj_forn_norm"].str.startswith("000").map(
    {True: "pessoa_fisica", False: "pessoa_juridica"}
)
print("Tipo de fornecedor:")
print(df["tipo_fornecedor"].value_counts())

COLS_CNPJ = [
    "cnpj_norm",
    "data_inicio_atividade",
    "porte",                      
    "natureza_juridica",         
    "municipio", "uf",
    "codigo_municipio_ibge",
    "cnae_fiscal",
    "capital_social",
    "descricao_situacao_cadastral",
    "opcao_pelo_mei",
    "opcao_pelo_simples",
]
cols_cnpj = [c for c in COLS_CNPJ if c in cnpj_df.columns]

# Deduplica o cadastro pela chave normalizada: identificadores que não são CNPJ (estrangeiros etc.) viram a mesma chave e
# repetiam cada proposta desses fornecedores no join. Mantém a linha sem erro, se houver.
n_cadastro = len(cnpj_df)
cnpj_unico = (cnpj_df.assign(_sem_erro=cnpj_df["erro"].isna() if "erro" in cnpj_df.columns else True)
              .sort_values("_sem_erro", ascending=False, kind="stable")
              .drop_duplicates("cnpj_norm", keep="first")
              .drop(columns="_sem_erro"))
print(f"Cadastro: {n_cadastro} linhas -> {len(cnpj_unico)} chaves normalizadas distintas")

n_antes_cnpj = len(df)
df = df.merge(
    cnpj_unico[cols_cnpj].rename(columns={"cnpj_norm": "cnpj_forn_norm"}),
    on="cnpj_forn_norm",
    how="left",
    validate="many_to_one"
)
assert len(df) == n_antes_cnpj, "o join com o cadastro não pode alterar o número de linhas"
n_sem_cnpj = (df["cnpj_forn_norm"] == "00000000000000").sum()
print(f"\nApós join com CNPJ: {len(df)} linhas (iguais às {n_antes_cnpj} de antes do join)")
print(f"Linhas com fornecedor sem CNPJ válido (chave 00000000000000): {n_sem_cnpj} ({n_sem_cnpj / len(df):.2%})")
print(f"porte nulos: {df['porte'].isna().sum() if 'porte' in df.columns else 'N/A'}")

Tipo de fornecedor:
tipo_fornecedor
pessoa_juridica    69446
pessoa_fisica       6056
Name: count, dtype: int64
Cadastro: 9458 linhas -> 9447 chaves normalizadas distintas

Após join com CNPJ: 75502 linhas (iguais às 75502 de antes do join)
Linhas com fornecedor sem CNPJ válido (chave 00000000000000): 70 (0.09%)
porte nulos: 18024


### 4. Join: + Coordenadas IBGE

In [5]:
municipios["codigoIbge"] = municipios["codigoIbge"].astype(str).str.strip()
municipios["latitude"]   = pd.to_numeric(municipios["latitude"],  errors="coerce")
municipios["longitude"]  = pd.to_numeric(municipios["longitude"], errors="coerce")

ibge_col = "unidadeOrgaoCodigoIbge"
if ibge_col in df.columns:
    df[ibge_col] = df[ibge_col].astype(str).str.strip()
    df = df.merge(
        municipios[["codigoIbge", "latitude", "longitude"]].rename(
            columns={"codigoIbge": ibge_col, "latitude": "lat_orgao", "longitude": "lon_orgao"}
        ),
        on=ibge_col,
        how="left"
    )

if "municipio" in df.columns:
    mun_coords = municipios[["nomeMunicipio", "latitude", "longitude"]].copy()
    mun_coords["nomeMunicipio"] = mun_coords["nomeMunicipio"].str.upper().str.strip()
    df["municipio_upper"] = df["municipio"].astype(str).str.upper().str.strip()
    df = df.merge(
        mun_coords.rename(columns={"nomeMunicipio": "municipio_upper", "latitude": "lat_forn", "longitude": "lon_forn"}),
        on="municipio_upper",
        how="left"
    )

print(f"Após joins de coordenadas: {len(df)} linhas")
print(f"lat_orgao nulos: {df['lat_orgao'].isna().sum() if 'lat_orgao' in df.columns else 'N/A'}")

Após joins de coordenadas: 75502 linhas
lat_orgao nulos: 0


### 5. Engenharia de Variaveis

In [6]:
preco_col = "valor_total_homologado"
df[preco_col] = pd.to_numeric(df[preco_col], errors="coerce")

df["valor_unitario_estimado"]   = pd.to_numeric(df["valor_unitario_estimado"],  errors="coerce")
df["valor_unitario_homologado"] = pd.to_numeric(df["valor_unitario_homologado"], errors="coerce")

df["desvio_item_estimado"] = (
    (df["valor_unitario_homologado"] - df["valor_unitario_estimado"])
    / df["valor_unitario_estimado"].replace(0, np.nan)
)

print(f"desvio_item_estimado — não nulos: {df['desvio_item_estimado'].notna().sum()}")
print(f"desvio_item_estimado — nulos: {df['desvio_item_estimado'].isna().sum()}")

desvio_item_estimado — não nulos: 74630
desvio_item_estimado — nulos: 872


#### Variáveis de preço descartadas

Duas variáveis de preço existiram na primeira versão deste notebook e foram **removidas em 02/09/2026**
(commit `fd14d86`), substituídas por `desvio_item_estimado`. A remoção não se deve a vazamento temporal de dados,
e sim a problemas de construção:

| Variável | Definição | Por que saiu |
|---|---|---|
| `desvio_preco_mediana` | `(valor − mediana do item) / mediana do item`, por órgão × compra × item | A API retorna só o vencedor: 90,7% dos itens têm uma única linha, então a mediana é o próprio valor. A variável vale 0 em 93,8% dos registros e não carrega informação. |
| `ratio_preco_estimado` | `valor homologado do item / valorTotalEstimado` **da contratação** | Numerador no nível do item e denominador no nível da compra: tem correlação de 0,75 (Spearman) com 1/número de itens da compra, ou seja, mede o tamanho da compra, não o preço. |

`desvio_item_estimado` compara o preço unitário vencedor com o preço unitário estimado no edital, ambos no nível do item.
O estimado é publicado antes da abertura das propostas.

In [7]:
# --- 5.2 Concorrência ---
n_propostas = df.groupby(
    ["orgaoEntidadeCnpj", "sequencialCompraPncp", "numeroItem"]
)["cnpj_forn_norm"].transform("count")

df["n_propostas_item"] = n_propostas
df["fornecedor_unico"] = (n_propostas == 1).astype(int)
print(f"Items com fornecedor único: {df['fornecedor_unico'].sum()} ({df['fornecedor_unico'].mean()*100:.1f}%)")

Items com fornecedor único: 68478 (90.7%)


In [8]:
# --- 5.3 Variaveis temporais ---
# Data de referência: PUBLICAÇÃO da contratação no PNCP (dataPublicacaoPncp). A data de abertura das propostas
# (dataAberturaPropostaPncp) não é usada: só está preenchida em 81% das linhas e difere da publicação em 26,6%.
df["data_publicacao"] = pd.to_datetime(df["dataPublicacaoPncp"], errors="coerce")

df["ano"] = df["data_publicacao"].dt.year
df["mes"] = df["data_publicacao"].dt.month
df["dia_semana"] = df["data_publicacao"].dt.dayofweek  # 0=segunda, 6=domingo
df["trimestre"] = df["data_publicacao"].dt.quarter
print("Variaveis temporais criadas")


Variaveis temporais criadas


#### Variáveis eleitorais

| Variável | Definição |
|---|---|
| `dias_desde_eleicao` | Dias entre a data de publicação e a eleição municipal de 06/10/2024 (1º turno): **negativo antes, positivo depois** (−641 a +85 na base). |
| `pre_eleitoral_180d` | 1 se publicada nos 180 dias imediatamente anteriores à eleição (−180 a −1): 40,3% da base. |
| `pos_eleitoral_180d` | 1 se publicada nos 180 dias seguintes à eleição (0 a +180): 19,2% da base; como a base termina em 30/12/2024, cobre só 85 dias. |

**Removidas:** `ano_eleitoral` (idêntica a `ano == 2024`, correlação 1,0) e a eleição de 2020 (nunca é a mais próxima).
A versão anterior de `pre_eleitoral_180d` usava a distância em módulo e marcava como "pré-eleitorais" 14.432 registros **posteriores** à eleição (32% dos marcados; contagem na base anterior à correção do join).

**Limitações:** (i) a janela anterior cai inteira entre abril e outubro de 2024, o que confunde janela, ano e mês: comparações devem ser feitas dentro de 2024; `ano` correlaciona 0,85 com a distância com sinal;
(ii) 75,9% dos registros são de órgãos federais (17,4% municipais e 6,6% estaduais, pelo campo `orgaoEntidadeEsferaId`), para os quais a eleição municipal tem pertinência limitada;
(iii) o 2º turno (27/10/2024, municípios com mais de 200 mil eleitores) não é considerado;
(iv) a data usada é a de publicação, e não a de abertura das propostas.

In [9]:
# --- 5.4 Variaveis eleitorais ---
# Eleição municipal de 2024, 1º turno (06/10/2024). A de 2020 não se aplica: a base cobre 2023-2024 e a eleição
# mais próxima é sempre a de 2024 (a menor distância até 2020 é de 780 dias).
ELEICAO = pd.Timestamp(2024, 10, 6)

# Distância com sinal, em dias: negativa ANTES da eleição, positiva DEPOIS.
df["dias_desde_eleicao"] = (df["data_publicacao"].dt.normalize() - ELEICAO).dt.days

# Janelas separadas e estritas (a antiga pre_eleitoral_180d usava |dias| <= 180 e incluía o período posterior).
df["pre_eleitoral_180d"] = ((df["dias_desde_eleicao"] >= -180) & (df["dias_desde_eleicao"] < 0)).astype(int)
df["pos_eleitoral_180d"] = ((df["dias_desde_eleicao"] >= 0) & (df["dias_desde_eleicao"] <= 180)).astype(int)
print(f"Propostas nos 180 dias ANTES da eleição:  {df['pre_eleitoral_180d'].sum()}")
print(f"Propostas nos 180 dias DEPOIS da eleição: {df['pos_eleitoral_180d'].sum()}")
print(f"Dias desde a eleição: min {df['dias_desde_eleicao'].min()}, max {df['dias_desde_eleicao'].max()}")


Propostas nos 180 dias ANTES da eleição:  30200
Propostas nos 180 dias DEPOIS da eleição: 14399
Dias desde a eleição: min -641, max 85


In [10]:
# --- 5.5 Distancia geografica fornecedor x orgao ---
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

if all(c in df.columns for c in ["lat_orgao", "lon_orgao", "lat_forn", "lon_forn"]):
    df["dist_km_forn_orgao"] = haversine(
        df["lat_forn"].astype(float),
        df["lon_forn"].astype(float),
        df["lat_orgao"].astype(float),
        df["lon_orgao"].astype(float)
    )
    df["fornecedor_fora_pe"] = (~df["uf"].fillna("").str.upper().eq("PE")).astype(int)
    print(f"Distancia media forn-orgao: {df['dist_km_forn_orgao'].mean():.1f} km")
else:
    print("Aviso: colunas de coordenadas nao encontradas, variavel de distancia nao criada")

Distancia media forn-orgao: 61.7 km


In [11]:
# --- 5.6 Variáveis da empresa ---
if "data_inicio_atividade" in df.columns:
    df["data_abertura_empresa"] = pd.to_datetime(df["data_inicio_atividade"], errors="coerce")
    df["idade_empresa_anos"] = (
        (df["data_publicacao"] - df["data_abertura_empresa"]).dt.days / 365.25
    ).clip(lower=0)
    df["empresa_nova"] = (df["idade_empresa_anos"] < 1).astype(int)
    print(f"Idade média das empresas: {df['idade_empresa_anos'].mean():.1f} anos")

# Frequência do fornecedor em licitações do órgão
# (quantas contratações distintas o fornecedor ganhou neste órgão)
freq_forn_orgao = df.groupby(
    ["orgaoEntidadeCnpj", "cnpj_forn_norm"]
)["sequencialCompraPncp"].transform("nunique")
df["freq_forn_orgao"] = freq_forn_orgao

print("Variáveis de empresa criadas")

Idade média das empresas: 12.2 anos
Variáveis de empresa criadas


### 6. Dataset Final

#### Valores homologados iguais a zero

Itens com `valor_unitario_homologado == 0` geram `desvio_item_estimado = -1`, que os modelos leriam como preço
extremamente baixo. Na base, são **747 linhas (0,99%)** e todas têm o mesmo perfil: **100% Dispensa** e **100% com
`situacao_resultado = "Cancelado"`**, distribuídas em 81 compras e 30 órgãos (733 federais). Nenhuma compra tem todos os
itens zerados. O padrão indica que o valor de um resultado cancelado é registrado como zero, e não que o preço
fosse baixo; por isso esses registros **saem da base de modelagem** e ficam em `data/output/registros_valor_zero.csv`
para consulta e auditoria.

Com os zeros na base, eles correspondiam a 19% do alto risco (contra 0,98% da base) e a 24% das sinalizações do Isolation Forest (medido antes da correção do join).
Marcá-los com uma variável indicadora piorou: o IF passou a sinalizar 48% de linhas zero.

**Mantidos e documentados como limitação:**
- 1.431 resultados cancelados com valor positivo: o desvio deles tem a mesma distribuição dos itens normais
  (mediana −0,20 contra −0,17), então o preço é válido.
- 872 itens com valor estimado igual a zero (866 na base de modelagem, depois de retirar 6 que também têm valor homologado zero; sem orçamento sigiloso): o desvio não é calculável e é imputado com 0,0 no NB06,
  o que o confunde com "cobrou exatamente o teto". Nenhum deles aparece no alto risco.

In [12]:
# --- 6.0 Registros com valor homologado zero ---
# Os 747 casos são todos itens de Dispensa com situacao_resultado == "Cancelado": o valor aparece zerado e o
# desvio_item_estimado = -1 não representa preço baixo. Saem da base de modelagem e ficam em arquivo próprio.
# A exclusão vem depois das demais variáveis de propósito: n_propostas_item e fornecedor_unico continuam contando
# as linhas canceladas (o fornecedor participou do item).
import os
zero_homologado = df["valor_unitario_homologado"] == 0

cols_registro = [c for c in [
    "orgaoEntidadeCnpj", "anoCompraPncp", "sequencialCompraPncp", "numeroItem", "cnpj_forn_norm",
    "nome_fornecedor", "modalidadeNome", "situacao_resultado", "descricaoItem",
    "valor_unitario_estimado", "valor_unitario_homologado", "valor_total_homologado"] if c in df.columns]

os.makedirs("data/output", exist_ok=True)
df.loc[zero_homologado, cols_registro].to_csv("data/output/registros_valor_zero.csv", index=False)

print(f"Registros com valor homologado zero: {zero_homologado.sum()} ({zero_homologado.mean()*100:.2f}%)")
print("  por modalidade:        ", df.loc[zero_homologado, "modalidadeNome"].value_counts().to_dict())
print("  por situacao_resultado:", df.loc[zero_homologado, "situacao_resultado"].value_counts().to_dict())
print("  com valor estimado também zero (desvio não calculável):",
      int((zero_homologado & (df["valor_unitario_estimado"] == 0)).sum()))
print("  salvos em data/output/registros_valor_zero.csv")

df = df.loc[~zero_homologado].reset_index(drop=True)
print(f"\nBase de modelagem após a exclusão: {len(df)} linhas")


Registros com valor homologado zero: 747 (0.99%)
  por modalidade:         {'Dispensa': 747}
  por situacao_resultado: {'Cancelado': 747}
  com valor estimado também zero (desvio não calculável): 6
  salvos em data/output/registros_valor_zero.csv



Base de modelagem após a exclusão: 74755 linhas


In [13]:
# Seleciona colunas finais para o modelo
FEATURES_ML = [
    # Identificadores
    "orgaoEntidadeCnpj", "anoCompraPncp", "sequencialCompraPncp", "numeroItem", "cnpj_forn_norm",
    # Preço
    "desvio_item_estimado",
    # Concorrência
    "n_propostas_item", "fornecedor_unico",
    # Temporal
    "ano", "mes", "trimestre", "dia_semana",
    # Eleitoral
    "dias_desde_eleicao", "pre_eleitoral_180d", "pos_eleitoral_180d",
    # Geográfica
    "dist_km_forn_orgao", "fornecedor_fora_pe",
    # Empresa
    "tipo_fornecedor", "idade_empresa_anos", "empresa_nova",
    "opcao_pelo_mei", "opcao_pelo_simples",
    # Contexto 
    "modalidadeNome", "porte", "cnae_fiscal", "descricao_situacao_cadastral"
]

cols_presentes = [c for c in FEATURES_ML if c in df.columns]
cols_ausentes  = [c for c in FEATURES_ML if c not in df.columns]

print(f"Features presentes ({len(cols_presentes)}): {cols_presentes}")
if cols_ausentes:
    print(f"Ausentes: {cols_ausentes}")

dataset_ml = df[cols_presentes].copy()
dataset_ml.to_csv("data/processed/dataset_ml.csv", index=False)
print(f"\nDataset salvo: {len(dataset_ml)} linhas x {len(dataset_ml.columns)} colunas")
print("\nEstatísticas:")
dataset_ml.describe()

Features presentes (26): ['orgaoEntidadeCnpj', 'anoCompraPncp', 'sequencialCompraPncp', 'numeroItem', 'cnpj_forn_norm', 'desvio_item_estimado', 'n_propostas_item', 'fornecedor_unico', 'ano', 'mes', 'trimestre', 'dia_semana', 'dias_desde_eleicao', 'pre_eleitoral_180d', 'pos_eleitoral_180d', 'dist_km_forn_orgao', 'fornecedor_fora_pe', 'tipo_fornecedor', 'idade_empresa_anos', 'empresa_nova', 'opcao_pelo_mei', 'opcao_pelo_simples', 'modalidadeNome', 'porte', 'cnae_fiscal', 'descricao_situacao_cadastral']



Dataset salvo: 74755 linhas x 26 colunas

Estatísticas:


,desvio_item_estimado,n_propostas_item,fornecedor_unico,ano,mes,trimestre,dia_semana,dias_desde_eleicao,pre_eleitoral_180d,pos_eleitoral_180d,dist_km_forn_orgao,fornecedor_fora_pe,idade_empresa_anos,empresa_nova
count,73889.000000,74755.000000,74755.000000,74755.000000,74755.000000,74755.000000,74755.00000,74755.000000,74755.000000,74755.000000,24296.000000,74755.000000,56754.000000,74755.000000
mean,-0.232783,1.245642,0.915965,2023.721731,7.449027,2.808347,2.03573,-169.849334,0.402528,0.192362,61.496177,0.628239,12.314400,0.068076
std,0.540357,2.025040,0.277441,0.448150,3.109416,0.999339,1.42936,173.107364,0.490411,0.394158,133.085663,0.483278,11.866038,0.251877
min,-1.000000,1.000000,0.000000,2023.000000,1.000000,1.000000,0.00000,-641.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-0.411072,1.000000,1.000000,2023.000000,5.000000,2.000000,1.00000,-305.000000,0.000000,0.000000,0.000000,0.000000,3.222450,0.000000
50%,-0.173048,1.000000,1.000000,2024.000000,8.000000,3.000000,2.00000,-137.000000,0.000000,0.000000,0.000000,1.000000,8.689938,0.000000
75%,-0.000145,1.000000,1.000000,2024.000000,10.000000,4.000000,3.00000,-31.000000,1.000000,0.000000,55.918158,1.000000,17.390828,0.000000
max,92.750000,42.000000,1.000000,2024.000000,12.000000,4.000000,6.00000,85.000000,1.000000,1.000000,640.924871,1.000000,116.331280,1.000000


In [14]:
missing = dataset_ml.isnull().sum()
missing_pct = (missing / len(dataset_ml) * 100).round(1)
relatorio = pd.DataFrame({"ausentes": missing, "pct": missing_pct})
print("Valores ausentes por coluna:")
print(relatorio[relatorio["ausentes"] > 0].sort_values("pct", ascending=False))

Valores ausentes por coluna:
                              ausentes   pct
dist_km_forn_orgao               50459  67.5
opcao_pelo_mei                   29039  38.8
opcao_pelo_simples               29039  38.8
idade_empresa_anos               18001  24.1
porte                            18001  24.1
cnae_fiscal                      18001  24.1
descricao_situacao_cadastral     18001  24.1
desvio_item_estimado               866   1.2
